# Qual2Cast: two comp pipelines

Run cells in order. **B3** and **B3 + LLM filter** share fresh profiles and
an evidence-only 0–3 judge. All training targets are included by default.

Choose `codex` (Sol fills/filter, Luna judge) or `claude` (Sonnet fills/filter,
Haiku judge). Preview is the default. Set `EXECUTE = True` below for live
model calls. Setup and output definitions: [README](README.md).


In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline/workflow.py").exists())
sys.path.insert(0, str(ROOT / "pipeline"))
from workflow import Workflow

PROVIDER = "codex"  # or "claude"
CONFIG = ROOT / "pipeline/configs" / f"{PROVIDER}.toml"
RUN_DIR = None  # paste a previous printed run directory here to resume it
EXECUTE = False  # True runs live model calls when executing the cells below

workflow = Workflow(CONFIG, RUN_DIR)
workflow.plan()


## 1. Freeze inputs
Save the verified company snapshot, original split and target list. Keep
financial fields separate. Every requested target stays in the final report,
including targets with no estimate.


In [ ]:
if EXECUTE:
    display(workflow.prepare().head())
else:
    print("Preview only. Set EXECUTE = True above and rerun from the setup cell to run all stages.")


## 2. Fill company facets with the stronger model
Fill all 533 companies, including the peer pool. The existing protocol uses
name/URL and model knowledge first, then saved website passages to fill unknown
facets. Failed gap fills retain earlier values and unknowns. Completed calls
are reused when resuming this run. New model fills need factual verification.


In [ ]:
if EXECUTE:
    facets = workflow.fill()
    display(facets[["IQ_COMPANY_ID", "business_id", "llm_status", "needs_review"]].head(10))


## 3. Build B3 candidates and the first pipeline
Blend embedding and lexical cosine per facet, combine with information weights,
exclude company size, and choose mutual top-10 peers topped up to three where
available. Unknown facets are missing. All peers come from training.


In [ ]:
if EXECUTE:
    candidates = workflow.rank()
    display(candidates.head(15))


## 4. Filter B3 peers: the second pipeline
Use the same profiles and candidates. The stronger selector receives no
similarity scores or financials. Save accepts, rejects and incomplete answers
separately. A completed target may have no accepted peers.


In [ ]:
if EXECUTE:
    decisions = workflow.filter()
    display(decisions[["target_ticker", "comp_ticker", "score", "reason", "filter_complete"]].head(15))


## 5. Judge every B3 pair with the weaker model
Run the existing evidence-only 0–3 rubric over **all** B3 pairs, not a sample.
The judge sees saved website passages, not the profiles, selector decisions or
financials. Reuse each pair's grade across both pipelines, including rejected
peers in the filter audit. Insufficient evidence is null, not zero. Grades do
not change the selected peers or valuation.


In [ ]:
if EXECUTE:
    pair_results = workflow.judge()
    display(pair_results[["target_ticker", "comp_ticker", "judge_score", "verdict", "judge_reason"]].head(15))


## 6. Compare quality, coverage and valuation
Use the median selected-peer multiple. Join the observed target multiple only
after matching to calculate error. Report coverage and common-target errors
alongside the judge distribution. Model assessments are not human accuracy.

`comparison/filter_audit.csv` includes reasons for both accepted and rejected
peers. Per-route `valuation.csv` keeps targets with no estimate. All paths are
under the printed provider/model/run directory. To analyze a completed saved
run after restarting, set `RUN_DIR`, recreate `workflow`, and call
`workflow.analyze()` directly; no model calls are made by that method.


In [ ]:
if EXECUTE:
    summary = workflow.analyze()
    display(summary)
    display(workflow.table("comparison/common_target_errors.csv").head())
    display(Image(filename=str(workflow.run_dir / "comparison/overview.png")))
    print("All outputs:", workflow.run_dir)
